# Word Tokenization in NLP

**Course:** Natural Language Processing · Unit 1 — Structural Foundations of Language  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 2 — Words and Tokens  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain what **tokenization** is and why it is the foundational step in any NLP pipeline
2. Tokenize text using **NLTK** `word_tokenize` (rule-based, Penn Treebank style)
3. Tokenize text using **spaCy** (neural, language-aware)
4. Compare the two approaches and identify when each is appropriate
5. Handle large corpora efficiently by chunking text before processing

## What is Tokenization?

Tokenization is the process of splitting raw text into discrete units called **tokens** — typically words, punctuation marks, or subwords. It is the very first step in virtually every NLP pipeline because downstream tasks (POS tagging, NER, parsing, classification) all operate on token sequences, not raw strings.

Different tokenizers make different choices:
- Should `"don't"` become `["do", "n't"]` or `["don't"]`?
- Are punctuation marks separate tokens?
- How are URLs, emails, or numbers handled?

## Dataset

This notebook processes the **Matilda corpus** — a collection of Spanish-language texts about women in engineering in Latin America, published by CONFEDI/LACCEI (2019). It is stored as `.txt` files in `data/matilda/`.

---

## 1. Environment Setup

In [ ]:
# Standard library
import glob
from pathlib import Path

# Third-party
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import spacy

# Resolve data directory relative to this notebook's location
# Works on any machine regardless of username or OS
DIR_DATA = Path.cwd() / 'data' / 'matilda'
print(f"Data directory: {DIR_DATA}")
print(f"Directory exists: {DIR_DATA.exists()}")

> **Note:** Using `pathlib.Path.cwd()` makes the path relative to the current working directory — no hardcoded usernames or OS-specific separators. If you move the notebook, just ensure `data/matilda/` is at the same level.

### 1.1 Download NLTK Resources

`punkt_tab` is the tokenizer model required by NLTK ≥ 3.9 (which supports Python 3.13).  
The older `punkt` resource is deprecated and will raise a `LookupError` on current installations.

In [ ]:
# Download required NLTK resources (only needed once per environment)
# punkt_tab replaces the deprecated punkt in NLTK >= 3.9 (Python 3.13 compatible)
nltk.download('punkt_tab', quiet=True)
print("NLTK resources ready.")

### 1.2 Load spaCy Language Model

If the model is not installed, run the following command in your terminal **once**:

```bash
python -m spacy download es_core_news_sm
```

In [ ]:
# Load the small Spanish spaCy model with graceful error handling
try:
    nlp = spacy.load("es_core_news_sm")
    print(f"spaCy model loaded: {nlp.meta['name']} v{nlp.meta['version']}")
except OSError:
    raise OSError(
        "spaCy model 'es_core_news_sm' not found.\n"
        "Install it by running: python -m spacy download es_core_news_sm"
    )

> **Output interpretation:** The print statement confirms which model and version were loaded. spaCy's Spanish small model (`es_core_news_sm`) uses a Convolutional Neural Network trained on AnCora and WikiNER data. It is compact (~12 MB) and supports tokenization, POS tagging, NER, and dependency parsing.

---

## 2. Load the Corpus

In [ ]:
# Discover all .txt files in the data directory
txt_files = sorted(DIR_DATA.glob("*.txt"))
print(f"Found {len(txt_files)} text file(s):")
for f in txt_files:
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

> **Output interpretation:** This cell inventories the corpus before loading it — a good practice to detect missing or unexpected files early. The file sizes give a rough sense of how much text each document contains.

In [ ]:
# Read and concatenate all files into a single string
corpus = ""
for filepath in txt_files:
    corpus += filepath.read_text(encoding="utf-8")

print(f"Total corpus length: {len(corpus):,} characters")
print(f"Approximate word count (rough): {len(corpus.split()):,}")

> **Output interpretation:** The character count gives the raw size of the corpus. The rough word count (splitting on whitespace) is a quick estimate — the precise token count from NLTK and spaCy will differ because they apply language-aware rules for punctuation, contractions, and special characters.

In [ ]:
# Preview the first 500 characters of the corpus
print(corpus[:500])

> **Output interpretation:** The preview shows the raw text as loaded — including possible whitespace characters (`\x0c` = form feed from PDF extraction), accented characters (á, é, ó), em dashes (–), and URLs. A tokenizer must handle all of these gracefully. Checking this preview before tokenizing is essential to catch encoding issues early.

---

## 3. Tokenization with NLTK

NLTK's `word_tokenize` uses the **Penn Treebank tokenizer** — a rule-based system that:
- Separates punctuation from words (`,`, `.`, `!`, `?`)
- Splits contractions (e.g., `"don't"` → `["do", "n't"]`)
- Handles abbreviations and sentence boundaries using the `punkt_tab` model

In [ ]:
# Tokenize the full corpus with NLTK
# language='spanish' activates Spanish-specific sentence boundary rules
nltk_tokens = word_tokenize(corpus, language='spanish')

print(f"Total NLTK tokens: {len(nltk_tokens):,}")
print(f"Unique NLTK types (vocabulary): {len(set(nltk_tokens)):,}")
print(f"\nFirst 30 tokens:")
print(nltk_tokens[:30])

> **Output interpretation:**
>
> - **Total tokens** counts every token including punctuation (`,`, `.`, `(`, `)`, `–`) and special characters — always higher than the rough whitespace split count.
> - **Unique types** is the **vocabulary size** (type-token ratio ≈ types/tokens). A ratio well below 1.0 indicates natural language's characteristic repetition; a higher ratio suggests a more varied or technical text.
> - The first 30 tokens show that NLTK correctly preserved Spanish accented characters and treated `(`, `)`, `-`, and URLs as separate tokens. Notice form-feed characters (`\x0c`) appear as tokens — these are PDF extraction artifacts that would typically be removed in a cleaning step before tokenization.

---

## 4. Tokenization with spaCy

spaCy's tokenizer uses a hybrid approach:
1. **Rule-based prefix/suffix/infix splitting** (punctuation, whitespace)
2. **Exception dictionaries** (contractions, abbreviations per language)
3. Neural model context for ambiguous cases

Unlike NLTK, spaCy processes text as a `Doc` object that preserves token metadata (part-of-speech, dependency, named entities).

**Important:** spaCy has a default text length limit of 1,000,000 characters. For larger corpora we split the text into chunks before processing.

In [ ]:
# Split the corpus into 500,000-character chunks to stay within spaCy's limit
CHUNK_SIZE = 500_000
chunks = [corpus[i:i + CHUNK_SIZE] for i in range(0, len(corpus), CHUNK_SIZE)]
print(f"Corpus split into {len(chunks)} chunk(s) of up to {CHUNK_SIZE:,} characters each.")

# Tokenize each chunk and collect all tokens
spacy_tokens = []
for i, chunk in enumerate(chunks):
    doc = nlp(chunk)
    spacy_tokens.extend([token.text for token in doc])
    print(f"  Chunk {i+1}: {len(doc):,} tokens")

print(f"\nTotal spaCy tokens: {len(spacy_tokens):,}")
print(f"Unique spaCy types (vocabulary): {len(set(spacy_tokens)):,}")
print(f"\nFirst 30 tokens:")
print(spacy_tokens[:30])

> **Output interpretation:**
>
> - **Chunk processing** avoids a `ValueError` from spaCy's internal length guard. Each chunk's token count is printed so you can verify no chunk was silently skipped.
> - **Comparison with NLTK:** spaCy typically produces a slightly different token count because it applies different rules for whitespace, newlines, and special characters. For example, spaCy may keep `\n\n` as a single whitespace token, while NLTK may split or discard it.
> - **First 30 tokens** — spaCy preserves form-feed and newline characters as tokens (you can see `\x0c`, `\n\n`). In a production pipeline, you would filter these out using `[token.text for token in doc if not token.is_space]`.

---

## 5. Comparison: NLTK vs spaCy

The cell below quantitatively compares the two tokenizers on this corpus.

In [ ]:
# Build a summary comparison table
comparison = pd.DataFrame({
    'Tokenizer': ['NLTK (word_tokenize)', 'spaCy (es_core_news_sm)'],
    'Total tokens': [len(nltk_tokens), len(spacy_tokens)],
    'Unique types': [len(set(nltk_tokens)), len(set(spacy_tokens))],
    'Type-Token Ratio': [
        round(len(set(nltk_tokens)) / len(nltk_tokens), 4),
        round(len(set(spacy_tokens)) / len(spacy_tokens), 4),
    ],
})
comparison

> **Output interpretation:**
>
> - Differences in token counts reflect each tokenizer's rules for whitespace, newlines, and special characters — neither is objectively correct; the right choice depends on the downstream task.
> - The **Type-Token Ratio (TTR)** measures lexical diversity. A TTR of ~0.15–0.25 is typical for Spanish corpora of this size. Lower TTR means more repetition (common in technical or formal writing); higher TTR suggests richer vocabulary.
> - **When to choose NLTK:** simpler projects, rule-based preprocessing, when you do not need metadata (POS, NER).
> - **When to choose spaCy:** when you need token metadata, lemmatization, or integration into a larger NLP pipeline. spaCy is also faster on large documents due to Cython optimisations.